From the given data set of online retail dataset, I would like to understand the customer behavior, product performance and revenue trends to get actionable insights to increase profitabilty and improve customer retention.
The main objectives for this projects would be:
1. Understand sales patterns
2. Identify top-performing products
3. Analyze customer purchasing behavior
4. Discover revenue trends
5. Segment customers using RFM Analysis
6. Generate actionable business recommendations

In [2]:
#Import Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# Loading Online Retail Dataset and returning first three rows

df_raw = pd.read_csv("OnlineRetail.csv", encoding="latin1")
df_raw.head(3)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom


Exploratory Data Analysis and Cleanup

In [4]:
# Getting basic information about dataset

df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 33.1 MB


In [5]:
#Invoice Date type changed from str to datetime

df_raw['InvoiceDate'] = pd.to_datetime(df_raw['InvoiceDate'])
df_raw['InvoiceDate'].dtype


dtype('<M8[us]')

In [6]:
df_raw.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


In [8]:
# There is negative value for quantity and unit price, which seems not correct, will have a more closer look at this
# We will first analyse the negative values for Quantity by applying filter on Qauntity to return only negative values
 
df_raw[(df_raw['Quantity']<0)].head()


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom


In [9]:
# From the above result for negative values we can see that it looks like the negative values are for cancelled order as shown by InvoiceNo preceding by C
# lets try to find out the following
# 1. How many rows of data has negative Quantity
# 2. How many rows of data start with letter C
# 3. Do all InvoiceNo starting with letter C has negative Quanity 
# 4. Are all negative quantity values are being explained with cancelled orders in InvoiceNo

negative_quantity_count =(df_raw['Quantity']<0).sum()
cancelled_orders_count = (df_raw['InvoiceNo'].str.startswith('C')).sum()
all_cancelled_are_negative = (df_raw.loc[df_raw['InvoiceNo'].str.startswith('C'), 'Quantity'] < 0).all()
all_negative_are_cancelled = (df_raw.loc[df_raw['Quantity'] < 0, 'InvoiceNo'].str.startswith('C')).all()

print("="*60)
print("Negative Quantity & Cancelled Orders Validation")
print("="*60)
print(f"Negative Quantity records           : {negative_quantity_count}")
print(f"Cancelled Invoice records           : {cancelled_orders_count}")
print(f"All cancelled orders are negative   : {all_cancelled_are_negative}")
print(f"All negative quantities cancelled   : {all_negative_are_cancelled}")
print("="*60)


Negative Quantity & Cancelled Orders Validation
Negative Quantity records           : 10624
Cancelled Invoice records           : 9288
All cancelled orders are negative   : True
All negative quantities cancelled   : False


In [10]:
#From the above report we can see that all cancelled orders are negative, but there are some negative orders which are not cancelled
#The count of those orders is 1336 records (Negative Quantity Records - Cancelled Invoice Orders)
#We will store these records in a new df "negative_not_cancelled" and analyse further
negative_not_cancelled = df_raw[(df_raw['Quantity'] < 0) &(~df_raw['InvoiceNo'].str.startswith('C'))]
negative_not_cancelled.info()


<class 'pandas.DataFrame'>
Index: 1336 entries, 2406 to 538919
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   InvoiceNo    1336 non-null   str           
 1   StockCode    1336 non-null   str           
 2   Description  474 non-null    str           
 3   Quantity     1336 non-null   int64         
 4   InvoiceDate  1336 non-null   datetime64[us]
 5   UnitPrice    1336 non-null   float64       
 6   CustomerID   0 non-null      float64       
 7   Country      1336 non-null   str           
dtypes: datetime64[us](1), float64(2), int64(1), str(4)
memory usage: 93.9 KB


In [12]:
#From above we can see that Negative Quantity values which are not cancelled donot have customer ID as well
#lets now have a look at the orders which have negative Unit price
negative_unit_price = df_raw[df_raw['UnitPrice']<0]
negative_unit_price.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


In [ ]:
#There are only two records for this and these are related to bad debt adjustment and since no customer ID as well, we will drop them from the table

In [18]:
# Lets have a quick look if there are any records which has zero unit price or zero quantity as they will have impact on the calculation for revenue

print(f"Total number of records with zero quantity: {(df_raw['Quantity'] == 0).sum()}")
print(f"Total number of records with zero unit price: {(df_raw['UnitPrice'] == 0).sum()}")

Total number of records with zero quantity: 0
Total number of records with zero unit price: 2515


In [35]:
print(df_raw['InvoiceDate'].is_unique)

False


In [39]:
# Unit price have 2515 records which have zero unit price, lets have a quick look at this, and lets sort it by the invoice date
zero_price_invoices = df_raw.loc[df_raw['UnitPrice'] == 0, 'InvoiceNo'].unique()
zero_price_invoices

<StringArray>
['536414', '536545', '536546', '536547', '536549', '536550', '536552',
 '536553', '536554', '536589',
 ...
 '581209', '581210', '581211', '581212', '581213', '581226', '581234',
 '581406', '581408', '581422']
Length: 2155, dtype: str

In [62]:
#From above we have the list of unique invoiceNo with zero unit price
#Lets have a look at all the records with these invoice number and we will sort them on the basis of invoice number
#We would like to see if there is any relationship between invoice number and these zero unit price to check if they were given out as part of free items
# Display all items on those invoices
zero_price_orders = (df_raw.loc[df_raw['InvoiceNo'].isin(zero_price_invoices),
        ['InvoiceNo','InvoiceDate','CustomerID','StockCode','Description','Quantity','UnitPrice']] .sort_values(['UnitPrice']))
zero_price_orders.info()

<class 'pandas.DataFrame'>
Index: 5237 entries, 7189 to 41448
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   InvoiceNo    5237 non-null   str           
 1   InvoiceDate  5237 non-null   datetime64[us]
 2   CustomerID   597 non-null    float64       
 3   StockCode    5237 non-null   str           
 4   Description  3783 non-null   str           
 5   Quantity     5237 non-null   int64         
 6   UnitPrice    5237 non-null   float64       
dtypes: datetime64[us](1), float64(2), int64(1), str(3)
memory usage: 327.3 KB


In [63]:
#I would like to understand the zero price orders to check how many of these invoice Number has some positive values for Unit price along with zero 

positive_price_invoices = zero_price_orders.loc[
    zero_price_orders['UnitPrice'] > 0,
    'InvoiceNo'
].unique()

positive_price_invoices

<StringArray>
['577696', '547417', '540372', '568567', '554037', '572893', '546406',
 '574138', '538071', '539263', '571035', '548871', '561284', '550188',
 '539722', '580366', '577314', '560283', '562973', '575579', '561669',
 '568158', '537197', '569716', '540832', '549100', '547396', '540355',
 '539750', '540356', '575748', '574920', '574252', '548318', '538877',
 '537640', '574469', '553000', '577129', '574175', '577168', '574879',
 '541109', '545160', '561916', '564530', '553521', '553539', '558340',
 '546933', '545176', '537534', '539856']
Length: 53, dtype: str

In [68]:
#Lets have a quick look at one of the records which has invoice no with both zero and positive Unit price
zero_price_orders[zero_price_orders['InvoiceNo'] =='547417']

,InvoiceNo,InvoiceDate,CustomerID,StockCode,Description,Quantity,UnitPrice
130188,547417,2011-03-23 10:25:00,13239.0,22062,CERAMIC BOWL WITH LOVE HEART DESIGN,36,0.00
130186,547417,2011-03-23 10:25:00,13239.0,21400,RED PUDDING SPOON,24,0.12
130187,547417,2011-03-23 10:25:00,13239.0,21401,BLUE PUDDING SPOON,24,0.12
130184,547417,2011-03-23 10:25:00,13239.0,21399,BLUE POLKADOT COFFEE MUG,48,0.39
130185,547417,2011-03-23 10:25:00,13239.0,21398,RED POLKADOT COFFEE MUG,48,0.39
130189,547417,2011-03-23 10:25:00,13239.0,37482P,CUBIC MUG PINK POLKADOT,6,0.39
130190,547417,2011-03-23 10:25:00,13239.0,84596F,SMALL MARSHMALLOWS PINK BOWL,24,0.42
130177,547417,2011-03-23 10:25:00,13239.0,37342,POLKADOT COFFEE CUP & SAUCER PINK,24,0.79
130195,547417,2011-03-23 10:25:00,13239.0,47559B,TEA TIME OVEN GLOVE,10,1.25
130191,547417,2011-03-23 10:25:00,13239.0,21232,STRAWBERRY CERAMIC TRINKET BOX,12,1.25


By analysing the data for the zero unit price, we could not find any rationale for the zero unit sales data and will pop for the following reasons:
1. Only 2515 records have this non zero values and a very small percentage of our full dataset
2. Zero-priced transactions do not represent revenue-generating sales and may distort sales volume if interpreted as purchases.
3. They contribute units but no revenue, potentially inflating product popularity without reflecting commercial performance.
4. Free or unexplained transactions can distort metrics such as average spend, basket value, and purchasing patterns.

In [69]:
#Remove duplicate values:
df_raw.drop_duplicates(inplace=True)
df_raw.info()

<class 'pandas.DataFrame'>
Index: 536641 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    536641 non-null  str           
 1   StockCode    536641 non-null  str           
 2   Description  535187 non-null  str           
 3   Quantity     536641 non-null  int64         
 4   InvoiceDate  536641 non-null  datetime64[us]
 5   UnitPrice    536641 non-null  float64       
 6   CustomerID   401604 non-null  float64       
 7   Country      536641 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), str(4)
memory usage: 36.8 MB


In [76]:
# Data Quality Summary

total_records = len(df_raw)
missing_customerid = df_raw['CustomerID'].isnull().sum()
negative_quantity = (df_raw['Quantity'] < 0).sum()
negative_unit_price = (df_raw['UnitPrice'] < 0).sum()
zero_unit_price = (df_raw['UnitPrice'] == 0).sum()
# Create summary table
data_quality_summary = pd.DataFrame({'Data Quality Metric': ['Total Records','Missing CustomerID','Negative Quantity','Negative Unit Price','Zero Unit Price'],
    'Count': [total_records, missing_customerid, negative_quantity, negative_unit_price, zero_unit_price]})

# Calculate percentage of total dataset
data_quality_summary['% of Dataset'] = (
    data_quality_summary['Count'] / total_records * 100
).round(2)

# Format percentage column
data_quality_summary['% of Dataset'] = (
    data_quality_summary['% of Dataset'].astype(str) + '%'
)

# Display the summary
data_quality_summary


,Data Quality Metric,Count,% of Dataset
0,Total Records,536641,100.0%
1,Missing CustomerID,135037,25.16%
2,Negative Quantity,10587,1.97%
3,Negative Unit Price,2,0.0%
4,Zero Unit Price,2510,0.47%


Data Cleanup:
From the report above, we can drop the following items as they will distort our analysis and the amount of data is also very less

1. Negative Quantity records
2. Negative Unit Price
3. Zero Unit Price items
4. 
Approx 25% percent of the data dont have customer ID, since it is significant part of data, we will retain these for Sales, Product and Revenue analysis since they represent valid purchases and contribute to business performance. However we will exclude them for doing the customer behavior and RFM analysis

So we will two cleaned data frame:
1. df_clean_sales: We will drop Negative quantity records and unit price along with zero unit price items, to be used for Sales, Revevuew and Product Analysis
2. df_clean_RFM: We will remove the null consumer id along with negative quantity and unit price and zero unit price items

In [77]:
df_clean_sales = df_raw[(df_raw['Quantity'] > 0) & (df_raw['UnitPrice'] >= 0)].copy()
df_clean_sales.info()

<class 'pandas.DataFrame'>
Index: 526052 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    526052 non-null  str           
 1   StockCode    526052 non-null  str           
 2   Description  525460 non-null  str           
 3   Quantity     526052 non-null  int64         
 4   InvoiceDate  526052 non-null  datetime64[us]
 5   UnitPrice    526052 non-null  float64       
 6   CustomerID   392732 non-null  float64       
 7   Country      526052 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), str(4)
memory usage: 36.1 MB


In [78]:
df_clean_RFM = df_clean_sales.dropna(subset=['CustomerID']).copy()
df_clean_RFM.info()

<class 'pandas.DataFrame'>
Index: 392732 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    392732 non-null  str           
 1   StockCode    392732 non-null  str           
 2   Description  392732 non-null  str           
 3   Quantity     392732 non-null  int64         
 4   InvoiceDate  392732 non-null  datetime64[us]
 5   UnitPrice    392732 non-null  float64       
 6   CustomerID   392732 non-null  float64       
 7   Country      392732 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), str(4)
memory usage: 27.0 MB
